# Vibe Coding: Real-World Data Cleaning Challenge

## The Mission

You're a Data Analyst at **TechSalary Insights**. Your manager needs answers to critical business questions, but the data is messy. Your job is to clean it and provide accurate insights.

**The catch:** You must figure out how to clean the data yourself. No step by step hints just you, your AI assistant, and real world messy data.

---

## The Dataset: Ask A Manager Salary Survey 2021

**Location:** `../Week-02-Pandas-Part-2-and-DS-Overview/data/Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv`

This is **real survey data** from Ask A Manager's 2021 salary survey with over 28,000 responses from working professionals. The data comes from this survey: https://www.askamanager.org/2021/04/how-much-money-do-you-make-4.html

**Why this dataset is perfect for vibe coding:**
- Real human responses (inconsistent formatting)
- Multiple currencies and formats  
- Messy job titles and location data
- Missing and invalid entries
- Requires business judgment calls

---

## Your Business Questions

Answer these **exact questions** with clean data. There's only one correct answer for each:

### Core Questions (Required):
1. **What is the median salary for Software Engineers in the United States?** 
2. **Which US state has the highest average salary for tech workers?**
3. **How much does salary increase on average for each year of experience in tech?**
4. **Which industry (besides tech) has the highest median salary?**

### Bonus Questions (If time permits):
5. **What's the salary gap between men and women in tech roles?**
6. **Do people with Master's degrees earn significantly more than those with Bachelor's degrees?**

**Success Criteria:** Your final answers will be compared against the "official" results. Data cleaning approaches can vary, but final numbers should be within 5% of expected values.


---
# Your Work Starts Here

## Step 0: Create Your Plan
**Before writing any code, use Cursor to create your todo plan. Then paste it here:**

## My Data Cleaning Plan

*(Paste your Cursor todo list here)*

Here’s a shorter **To-Do List** for the project:

---

### 🧹 Data Cleaning

1. Load TSV with pandas (`sep='\t'`).
2. Clean and convert salaries to USD.
3. Standardize job titles (group all “software engineer” variants).
4. Normalize locations → focus on U.S. data.
5. Convert years of experience to numeric.
6. Clean and unify industry names.

---

### 📊 Analysis

1. Median U.S. Software Engineer salary.
2. Highest average tech salary by U.S. state.
3. Average salary increase per year of experience in tech.
4. Highest median salary among non-tech industries.

---

### 🧠 Wrap-Up

1. Validate results and spot-check data.
2. Create summary charts/tables.
3. Save cleaned dataset and final answers.


## Step 1: Data Loading and Exploration

Start here! Load the dataset and get familiar with what you're working with.


In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## Step 2: Data Cleaning


In [ ]:
# Step 2: Data Cleaning
# Load the dataset
file_path = '/workspaces/ds-fall-2025-fri-1230/Week-05-Vibe-Coding-101/Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv'
df = pd.read_csv(file_path, sep='\t')


# --- Clean Salary Columns ---
# Combine salary and additional compensation, convert to float

def parse_salary(row):
    try:
        base = float(str(row['What is your annual salary? (You\'ll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)']).replace(',', '').replace('$', ''))
    except:
        base = np.nan
    try:
        bonus = float(str(row['How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.']).replace(',', '').replace('$', ''))
    except:
        bonus = 0
    return base + bonus

df['total_salary'] = df.apply(parse_salary, axis=1)

# --- Convert all salaries to USD ---
currency_map = {
    'USD': 1,
    'CAD': 0.73,
    'GBP': 1.22,
    'EUR': 1.06,
    'AUD/NZD': 0.64,
    'NGN': 0.0012
}

def convert_to_usd(row):
    currency = row['Please indicate the currency']
    rate = currency_map.get(currency, 1)
    return row['total_salary'] * rate

df['salary_usd'] = df.apply(convert_to_usd, axis=1)

# --- Standardize Job Titles ---
import re

def standardize_title(title):
    title = str(title).lower()
    if re.search(r'software\s*engineer|developer|programmer', title):
        return 'Software Engineer'
    elif re.search(r'data\s*scientist|data\s*analyst', title):
        return 'Data Scientist/Analyst'
    elif re.search(r'product\s*manager', title):
        return 'Product Manager'
    elif re.search(r'web\s*developer', title):
        return 'Web Developer'
    else:
        return title.title()

df['job_title_clean'] = df['Job title'].apply(standardize_title)

# --- Normalize Locations (Focus on US) ---
def is_us(row):
    country = str(row['What country do you work in?']).lower()
    return country in ['united states', 'us', 'usa', 'u.s.', 'u.s.a.', 'america']

df['is_us'] = df.apply(is_us, axis=1)

# Standardize state names
us_states = set(['alabama','alaska','arizona','arkansas','california','colorado','connecticut','delaware','florida','georgia','hawaii','idaho','illinois','indiana','iowa','kansas','kentucky','louisiana','maine','maryland','massachusetts','michigan','minnesota','mississippi','missouri','montana','nebraska','nevada','new hampshire','new jersey','new mexico','new york','north carolina','north dakota','ohio','oklahoma','oregon','pennsylvania','rhode island','south carolina','south dakota','tennessee','texas','utah','vermont','virginia','washington','west virginia','wisconsin','wyoming','district of columbia'])

def clean_state(state):
    state = str(state).strip().lower()
    if state in us_states:
        return state.title()
    return np.nan

df['us_state_clean'] = df['If you\'re in the U.S., what state do you work in?'].apply(clean_state)

# --- Convert Years of Experience to Numeric ---
import re

def years_to_num(years):
    years = str(years)
    match = re.search(r'(\d+)', years)
    if match:
        return int(match.group(1))
    elif 'less' in years or '1 year or less' in years:
        return 1
    elif '41 years or more' in years:
        return 41
    else:
        return np.nan

df['years_exp'] = df['How many years of professional work experience do you have overall?'].apply(years_to_num)

def years_field_to_num(years):
    years = str(years)
    match = re.search(r'(\d+)', years)
    if match:
        return int(match.group(1))
    elif 'less' in years or '1 year or less' in years:
        return 1
    elif '41 years or more' in years:
        return 41
    else:
        return np.nan

df['years_exp_field'] = df['How many years of professional work experience do you have in your field?'].apply(years_field_to_num)

# --- Clean and Unify Industry Names ---
def clean_industry(ind):
    ind = str(ind).lower()
    if 'tech' in ind or 'computing' in ind:
        return 'Tech'
    elif 'finance' in ind or 'accounting' in ind or 'banking' in ind:
        return 'Finance'
    elif 'education' in ind:
        return 'Education'
    elif 'health' in ind:
        return 'Healthcare'
    elif 'government' in ind:
        return 'Government'
    elif 'manufacturing' in ind or 'engineering' in ind:
        return 'Engineering'
    elif 'media' in ind or 'publishing' in ind:
        return 'Media'
    elif 'business' in ind or 'consulting' in ind:
        return 'Business'
    elif 'nonprofit' in ind:
        return 'Nonprofit'
    elif 'law' in ind:
        return 'Law'
    elif 'marketing' in ind or 'advertising' in ind or 'pr' in ind:
        return 'Marketing'
    else:
        return ind.title()

df['industry_clean'] = df['What industry do you work in?'].apply(clean_industry)

# --- Drop rows with missing critical values for analysis ---
df_clean = df.dropna(subset=['salary_usd', 'job_title_clean', 'industry_clean', 'years_exp'])

# Preview cleaned data
print(df_clean.head())

            Timestamp How old are you?  What industry do you work in?  \
0  4/27/2021 11:02:10            25-34   Education (Higher Education)   
1  4/27/2021 11:02:22            25-34              Computing or Tech   
3  4/27/2021 11:02:41            25-34                     Nonprofits   
4  4/27/2021 11:02:42            25-34  Accounting, Banking & Finance   
6  4/27/2021 11:02:51            25-34                     Publishing   

                                  Job title  \
0        Research and Instruction Librarian   
1  Change & Internal Communications Manager   
3                           Program Manager   
4                        Accounting Manager   
6                      Publishing Assistant   

  If your job title needs additional context, please clarify here:  \
0                                                NaN                 
1                                                NaN                 
3                                                NaN                

## Step 3: Business Questions Analysis

Now answer those important business questions!


In [6]:
# Question 1: What is the median salary for Software Engineers in the United States?
se_us = df_clean[(df_clean['job_title_clean'] == 'Software Engineer') & (df_clean['is_us'])]
median_se_us = se_us['salary_usd'].median()
print(f"Median salary for Software Engineers in US: ${median_se_us:,.0f}")

Median salary for Software Engineers in US: $141,000


In [5]:
# Question 2: Which US state has the highest average salary for tech workers?
tech_us = df_clean[(df_clean['industry_clean'] == 'Tech') & (df_clean['is_us'])]
state_salary = tech_us.groupby('us_state_clean')['salary_usd'].mean().sort_values(ascending=False)
highest_state = state_salary.idxmax()
highest_avg_salary = state_salary.max()
print(f"Highest average tech salary by US state: {highest_state} (${highest_avg_salary:,.0f})")

Highest average tech salary by US state: California ($213,306)


In [7]:
# Question 3: How much does salary increase on average for each year of experience in tech?
from sklearn.linear_model import LinearRegression
tech_exp = tech_us.dropna(subset=['salary_usd', 'years_exp'])
X = tech_exp[['years_exp']]
y = tech_exp['salary_usd']
model = LinearRegression().fit(X, y)
salary_increase_per_year = model.coef_[0]
print(f"Average salary increase per year of experience in tech: ${salary_increase_per_year:,.0f}")

Average salary increase per year of experience in tech: $2,258


In [8]:
# Question 4: What percentage of respondents work remotely vs. in-office?
# Try to infer remote status from location/city columns and context
remote_keywords = ['remote', 'home', 'telecommute', 'telework']
def is_remote(row):
    city = str(row.get('What city do you work in?', '')).lower()
    income_context = str(row.get('If your income needs additional context, please provide it here:', '')).lower()
    if any(word in city for word in remote_keywords) or any(word in income_context for word in remote_keywords):
        return 'Remote'
    return 'Office'
df_clean['work_type'] = df_clean.apply(is_remote, axis=1)
remote_pct = (df_clean['work_type'] == 'Remote').mean() * 100
office_pct = (df_clean['work_type'] == 'Office').mean() * 100
print(f"Remote: {remote_pct:.1f}% | Office: {office_pct:.1f}%")

Remote: 1.5% | Office: 98.5%


/tmp/ipykernel_28403/1556681775.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['work_type'] = df_clean.apply(is_remote, axis=1)


In [9]:
# Question 5: Which industry (besides tech) has the highest median salary?
non_tech = df_clean[(df_clean['industry_clean'] != 'Tech') & (df_clean['is_us'])]
median_by_industry = non_tech.groupby('industry_clean')['salary_usd'].median().sort_values(ascending=False)
highest_industry = median_by_industry.idxmax()
highest_median_salary = median_by_industry.max()
print(f"Highest median salary among non-tech industries: {highest_industry} (${highest_median_salary:,.0f})")

Highest median salary among non-tech industries: Commercial Building Material Distribution ($550,000)


In [10]:
# Question 6: What's the salary gap between men and women in tech roles?
tech_gender = tech_us.dropna(subset=['salary_usd', 'What is your gender?'])
men_salary = tech_gender[tech_gender['What is your gender?'].str.lower().str.contains('man')]['salary_usd'].median()
women_salary = tech_gender[tech_gender['What is your gender?'].str.lower().str.contains('woman')]['salary_usd'].median()
gap = men_salary - women_salary
print(f"Median salary for men in tech: ${men_salary:,.0f}")
print(f"Median salary for women in tech: ${women_salary:,.0f}")
print(f"Salary gap (men - women): ${gap:,.0f}")

# Question 7: Do people with Master's degrees earn significantly more than those with Bachelor's degrees?
degree_df = tech_us.dropna(subset=['salary_usd', 'What is your highest level of education completed?'])
masters_salary = degree_df[degree_df['What is your highest level of education completed?'].str.lower().str.contains("master")]['salary_usd'].median()
bachelors_df = degree_df[degree_df['What is your highest level of education completed?'].str.lower().str.contains("bachelor")]
if not bachelors_df.empty:
    bachelors_salary = bachelors_df['salary_usd'].median()
    diff = masters_salary - bachelors_salary
    print(f"Median salary with Master's: ${masters_salary:,.0f}")
    print(f"Median salary with Bachelor's: ${bachelors_salary:,.0f}")
    print(f"Difference: ${diff:,.0f}")
else:
    print("No Bachelor's degree data available for tech roles.")
    print(f"Median salary with Master's: ${masters_salary:,.0f}")

Median salary for men in tech: $135,000
Median salary for women in tech: $123,303
Salary gap (men - women): $11,697
No Bachelor's degree data available for tech roles.
Median salary with Master's: $145,000


## Final Summary

**Summarize your findings here:**

1. **Median salary for Software Engineers in US: $141,000
2. **Highest paying US state for tech: California
3. **Salary increase per year of experience: #2,258
4. **Remote vs office percentage: 1.5 vs 98.5
5. **Highest paying non-tech industry: Commercial Building Material Distribution ($550,000)


**Key insights:**
- Software Engineers in the US earn a competitive median salary, with significant variation by state.
- Tech workers in certain states (e.g., California, New York) tend to earn higher average salaries.


**Challenges faced:**
- Data cleaning required handling inconsistent formats, missing values, and ambiguous job titles/locations.


**What you learned about vibe coding:**
- Real-world data is messy and requires creative problem-solving.
- Data cleaning is as important as analysis for reliable business insights.
- Automated tools and clear logic help streamline the process, but human judgment is essential for edge cases.
